# Au+Au 200 GeV bulk with the O+O chain, against PHOBOS

**Why.** Before the O+O chain's jet-quenching setting is calibrated in Au+Au (tuning page, "A3 and
the proposed runs"), its Au+Au bulk has to be right. The RHIC bulk tune changed only the
3dMCGlauber block, on p+p, O+O and d+Au. MUSIC's settings are the JETSCAPE MAP of 2601.17234,
which was calibrated on Au+Au, except that our freeze-out is at T = 0.15 GeV and there is no
afterburner.

**The model** (2026-11-25, `/data/AuAu_isr_smoke/bulk`): the O+O physics-run chain
(`OO_MCGlauber_MUSIC_isr_rhictune.xml`: 3dMCGlauber with the RHIC bulk tune, PythiaIsrGun +
iMATTER, MUSIC, iSS in the job) with
- `--xml-set` Au projectile and target, generated nucleon positions, MUSIC grid ±13 fm;
- minimum-bias-like settings as for `phys_mb`: `--no-subtraction --hard-pair-string-veto 0`,
  p̂_T 9–11 GeV;
- two impact-parameter classes, with geometric edges (σ_AuAu ≈ 6.9 b): 0–6 % (b < 3.6 fm) and
  40–50 % (b = 9.4–10.5 fm), 150 events each;
- charged hadrons: the iSS bulk only, |η| < 5.6. With `--no-subtraction` the strings carry the full
  nucleon energy and the hard event's hadrons come on top, so they are not counted (as `oo_bias.load`
  does for the minimum-bias samples).

**The data**: PHOBOS, arXiv:1011.1940 (INSPIRE 876609), Au+Au 200 GeV.
- Table V: dN_ch/dη at |η| < 1 per centrality bin.
- Table VII: N_ch within |η| < 5.4, and the FWHM.
- The bins here average PHOBOS's 0–3 + 3–6 % and 40–45 + 45–50 %. 45–50 % is PHOBOS's most
  peripheral bin with the full η range.
- The per-η points (Fig. 18; the tables of values are only in the paper's EPAPS supplement, and no
  HEPData record was found) are read from the paper's vector figure by
  `refs/phobos_auau200/extract_phobos_auau200.py` → `phobos_auau200_dndeta.json`: 54 points per
  bin, |η| < 5.3 in steps of 0.2, with the 90 % C.L. systematic band. Check against the paper's
  own tables: the integral of the points gives N_ch(|η| < 5.4) of Table VII within 0.3–0.6 % in all
  11 bins, the mean at |η| < 1 the tracklet values of Table V within 1–3 %. The statistical bars are
  smaller than the markers.

**Caveats.**
- PHOBOS classes are multiplicity classes; ours are impact-parameter classes. The comparison per
  participant pair, dN/dη/(N_part/2), does not depend on that.
- PHOBOS subtracted weak-decay daughters. iSS decays K0S and Λ here, so a few % excess is expected.
- Histograms are `hist` objects; the errors are statistical (the event-by-event spread / √N_ev).
- **Model results, not vetted.**

In [ ]:
import sys as _sys
_sys.path.insert(0, "..")
import _paths  # noqa: E402,F401  (analysis/OO-RHIC and its test directories on sys.path; shared results/, refs/, data/ in ..)
import glob, json, sys
from pathlib import Path
import numpy as np
import h5py, hdf5plugin  # noqa: F401
import hist
import matplotlib.pyplot as plt
sys.path.insert(0, ".")
import oo_bias as B

ROOT = Path("/data/AuAu_isr_smoke/bulk")
CH = B.charged_ids()
CLASSES = {"c0_6": "0-6 %", "c40_50": "40-50 %"}
# PHOBOS, 200 GeV: Table V (mid-rapidity, tracklets, 90 % C.L. syst.) and Table VII (|eta| < 5.4)
T5 = [("0-3", 361, 691, 52), ("3-6", 331, 619, 46), ("6-10", 297, 540, 41), ("10-15", 255, 465, 35),
      ("15-20", 215, 384, 29), ("20-25", 180, 313, 24), ("25-30", 150, 257, 19), ("30-35", 124, 208, 16),
      ("35-40", 101, 165, 12), ("40-45", 82, 133, 10), ("45-50", 65, 100, 8), ("50-55", 49, 73, 5),
      ("55-60", 37, 54, 4), ("60-65", 28, 38, 3), ("65-70", 20, 27, 2)]
T7 = {"0-3": (5159, 258, 7.63), "3-6": (4753, 238, 7.64), "40-45": (1096, 55, 8.11), "45-50": (847, 42, 8.23)}
PH = {"c0_6": ("0-3", "3-6"), "c40_50": ("40-45", "45-50")}
def phobos(cls):
    a, b = PH[cls]; r5 = {r[0]: r for r in T5}
    return dict(npart=(r5[a][1] + r5[b][1]) / 2, mid=(r5[a][2] + r5[b][2]) / 2, mid_e=(r5[a][3] + r5[b][3]) / 2,
                nch=(T7[a][0] + T7[b][0]) / 2, nch_e=(T7[a][1] + T7[b][1]) / 2, fwhm=(T7[a][2] + T7[b][2]) / 2)

def eta_of(p):
    pt = np.hypot(p[:, 1], p[:, 2]); pa = np.sqrt(pt ** 2 + p[:, 3] ** 2)
    return 0.5 * np.log(np.clip(pa + p[:, 3], 1e-12, None) / np.clip(pa - p[:, 3], 1e-12, None))

AX = hist.axis.Regular(56, -5.6, 5.6, name="eta", label=r"$\eta$")
def load(cls, root=ROOT):
    # per class: one hist of all charged bulk hadrons (Weight storage) and, per event, a row of counts
    # (for the event-by-event error); N_part per event.  Bulk only (--no-subtraction samples).
    h_all = hist.Hist(AX, storage=hist.storage.Weight())
    rows, npart = [], []
    for bf in sorted(glob.glob(str(root / cls / "*_bulk_bg.h5"))):
        with h5py.File(bf) as hb:
            n = int(hb.attrs["nunits_written"]); pb, cb = B._units(hb, n); npart.append(hb["units/glauber_npart"][:n])
        ob = np.r_[0, np.cumsum(cb)[:-1]]
        for e in range(n):
            eta = eta_of(pb[ob[e]:ob[e] + cb[e]])
            h_all.fill(eta=eta)
            rows.append(hist.Hist(AX).fill(eta=eta).values())
    return h_all, np.array(rows, float), np.concatenate(npart) if npart else np.zeros(0)

DATA = {cls: load(cls) for cls in CLASSES}
ROOT_MAP = Path("/data/AuAu_isr_smoke/bulk_map")   # the same with the 2601.17234 MAP 3dMCGlauber block
DATA_MAP = {cls: load(cls, ROOT_MAP) for cls in CLASSES} if ROOT_MAP.exists() else {}
WIDTH = AX.widths
for cls, (h, rows, npart) in DATA.items():
    print(f"{CLASSES[cls]}: {len(rows)} events, <N_part> {npart.mean():.0f} (PHOBOS {phobos(cls)['npart']:.0f}), "
          f"{h.sum().value / max(len(rows), 1):.0f} charged hadrons per event in |eta| < 5.6")

## dN_ch/dη over the full η range

Points: the model (statistical errors from the event-by-event spread). Open squares and grey band:
PHOBOS's points of Fig. 18 with their 90 % C.L. systematic band, the class as the average of its
two bins. Dark box: PHOBOS's tracklet value at |η| < 1 (Table V). Dashed: half of the model's maximum, at
PHOBOS's FWHM, i.e. where PHOBOS's distribution falls to half its height. For 0–6 % also
PHOBOS's limiting-fragmentation line for 0–3 %, dN/dη = 205 (y_beam + 0.11 − |η|) for
2 < |η| < 5, scaled to 0–6 % by the mid-rapidity ratio.

In [ ]:
import json
PJ = json.loads(Path("../refs/phobos_auau200/phobos_auau200_dndeta.json").read_text())["bins"]
def phobos_pts(cls):
    # the class as the average of PHOBOS's two bins (same eta points): eta, value, sys lo, sys hi
    a, b = PH[cls]
    A, Bb = PJ[a]["points"], PJ[b]["points"]
    eta = np.array([0.5 * (p["eta"] + q["eta"]) for p, q in zip(A, Bb)])
    val = np.array([0.5 * (p["value"] + q["value"]) for p, q in zip(A, Bb)])
    lo = np.array([0.5 * (p["sys_lo"] + q["sys_lo"]) for p, q in zip(A, Bb)])
    hi = np.array([0.5 * (p["sys_hi"] + q["sys_hi"]) for p, q in zip(A, Bb)])
    return eta, val, lo, hi

fig, axs = plt.subplots(2, 2, figsize=(14, 7.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
ax, axr = axs[0], axs[1]
x = AX.centers
SUMMARY = {}
for j, (cls, (h, rows, npart)) in enumerate(DATA.items()):
    if not len(rows):
        continue
    m = rows.mean(0) / WIDTH; e = rows.std(0) / np.sqrt(len(rows)) / WIDTH
    ph = phobos(cls)
    ax[j].errorbar(x, m, e, 0.5 * WIDTH, fmt="o", ms=3, color="C3", label=f"model, RHIC bulk tune, {len(rows)} events")
    ax[j].fill_between([-1, 1], ph["mid"] - ph["mid_e"], ph["mid"] + ph["mid_e"], color="k", alpha=0.2, label="PHOBOS |η|<1 (90 % CL)")
    half = m.max() / 2
    ax[j].plot([-ph["fwhm"] / 2, ph["fwhm"] / 2], [half, half], "k--", lw=1, label=f"PHOBOS FWHM {ph['fwhm']:.2f}")
    if cls == "c0_6":
        ybeam = 5.361; eta_lf = np.linspace(2, 5, 30); scale = ph["mid"] / 691
        lf = 205 * (ybeam + 0.11 - eta_lf) * scale
        ax[j].plot(eta_lf, lf, "k:", lw=1.2, label="PHOBOS limiting fragmentation (0-3 %, scaled)")
        ax[j].plot(-eta_lf, lf, "k:", lw=1.2)
    if cls in DATA_MAP and len(DATA_MAP[cls][1]):
        rm = DATA_MAP[cls][1]; mm = rm.mean(0) / WIDTH; em = rm.std(0) / np.sqrt(len(rm)) / WIDTH
        ax[j].errorbar(x, mm, em, 0.5 * WIDTH, fmt="^", ms=3, color="C0", alpha=0.8, label=f"model, MAP 3dMCGlauber block, {len(rm)} events")
    pe, pv, plo, phi_ = phobos_pts(cls)
    ax[j].fill_between(pe, pv - plo, pv + phi_, color="0.6", alpha=0.35, lw=0, step="mid")
    ax[j].plot(pe, pv, "s", ms=3, mfc="none", color="k", label="PHOBOS (Fig. 18; band: 90 % CL syst.)")
    mi = np.interp(pe, x, m); r = mi / pv
    axr[j].fill_between(pe, 1 - plo / pv, 1 + phi_ / pv, color="0.6", alpha=0.35, lw=0, step="mid", label="PHOBOS 90 % CL")
    axr[j].errorbar(pe, r, np.interp(pe, x, e) / pv, fmt="o", ms=3, color="C3", label="RHIC tune / PHOBOS")
    if cls in DATA_MAP and len(DATA_MAP[cls][1]):
        axr[j].errorbar(pe, np.interp(pe, x, mm) / pv, np.interp(pe, x, em) / pv, fmt="^", ms=3, color="C0", alpha=0.8, label="MAP block / PHOBOS")
        rmap = np.interp(pe, x, mm) / pv
        print(f"{CLASSES[cls]}: MAP block/PHOBOS at |eta| = " + ", ".join(f"{s_:g}: {np.mean(rmap[np.abs(np.abs(pe) - s_) < 0.15]):.2f}" for s_ in (0.1, 1, 2, 3, 4, 4.5, 5, 5.3))
              + f"; <N_part> {DATA_MAP[cls][2].mean():.0f}")
    axr[j].axhline(1, color="k", lw=0.8); axr[j].set_ylim(0.6, 1.6); axr[j].set_xlabel(r"$\eta$"); axr[j].legend(fontsize=7)
    print(f"{CLASSES[cls]}: model/PHOBOS at |eta| = " + ", ".join(f"{s_:g}: {np.mean(r[np.abs(np.abs(pe) - s_) < 0.15]):.2f}" for s_ in (0.1, 1, 2, 3, 4, 4.5, 5, 5.3)))
    ax[j].set_title(f"Au+Au 200 GeV, {CLASSES[cls]}", fontsize=10); ax[j].set_ylabel(r"$dN_{ch}/d\eta$")
    ax[j].set_ylim(0, None); ax[j].legend(fontsize=7)
    # summary numbers
    mid = m[np.abs(x) < 1].mean(); nch = (m * WIDTH)[np.abs(x) < 5.4].sum()
    above = np.where(m >= half)[0]; fw = x[above[-1]] - x[above[0]] + WIDTH[0]
    SUMMARY[cls] = dict(mid=mid, mid_ratio=mid / ph["mid"], pp=mid / (npart.mean() / 2), pp_ph=ph["mid"] / (ph["npart"] / 2),
                        pp_ratio=(mid / npart.mean()) / (ph["mid"] / ph["npart"]), nch=nch, nch_ratio=nch / ph["nch"], fwhm=fw, fwhm_ph=ph["fwhm"])
fig.suptitle("Figure 1. Charged-particle dN/dη: the O+O chain in Au+Au against PHOBOS (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
print(f"{'class':8s} {'dN/deta |eta|<1':>16s} {'/PHOBOS':>8s} {'per N_part/2':>13s} {'PHOBOS':>7s} {'ratio':>6s} {'N_ch |eta|<5.4':>15s} {'/PHOBOS':>8s} {'FWHM':>6s} {'PHOBOS':>7s}")
for cls, s in SUMMARY.items():
    print(f"{CLASSES[cls]:8s} {s['mid']:16.0f} {s['mid_ratio']:8.3f} {s['pp']:13.2f} {s['pp_ph']:7.2f} {s['pp_ratio']:6.3f} {s['nch']:15.0f} {s['nch_ratio']:8.3f} {s['fwhm']:6.2f} {s['fwhm_ph']:7.2f}")

## Centrality dependence per participant pair

PHOBOS's dN/dη(|η| < 1)/(N_part/2) for all 15 bins (Table V), against the model's two classes at their
mean N_part. This comparison is free of how the classes are made. Right: the model's per-event
mid-rapidity multiplicity against N_part (a 2D `hist`), with PHOBOS's bins.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
npp = np.array([r[1] for r in T5]); mid = np.array([r[2] for r in T5]); mide = np.array([r[3] for r in T5])
ax[0].errorbar(npp, mid / (npp / 2), mide / (npp / 2), fmt="s", color="k", mfc="none", label="PHOBOS 200 GeV (Table V)")
H2 = hist.Hist(hist.axis.Regular(40, 0, 400, name="npart", label=r"$N_{part}$"),
               hist.axis.Regular(50, 0, 1000, name="mid", label=r"$dN_{ch}/d\eta$, $|\eta|<1$"))
for cls, (h, rows, npart) in DATA.items():
    if not len(rows):
        continue
    mid_ev = rows[:, np.abs(AX.centers) < 1].sum(1) / 2.0
    pp = mid_ev / (npart / 2)
    ax[0].errorbar(npart.mean(), pp.mean(), pp.std() / np.sqrt(len(pp)), npart.std(), fmt="o", ms=7, color="C3",
                   label=f"model {CLASSES[cls]}")
    H2.fill(npart=npart, mid=mid_ev)
ax[0].set_xlabel(r"$\langle N_{part}\rangle$"); ax[0].set_ylabel(r"$dN_{ch}/d\eta\,/\,(N_{part}/2)$, $|\eta|<1$")
ax[0].set_ylim(0, 5); ax[0].legend(fontsize=8)
H2.plot2d(ax=ax[1], cmap="viridis", cmin=1)
ax[1].errorbar(npp, mid, mide, fmt="s", color="w", mec="k", mfc="none", ms=5, label="PHOBOS bins")
ax[1].legend(fontsize=8)
fig.suptitle("Figure 2. Mid-rapidity multiplicity per participant pair, and per event against N_part", fontsize=11)
fig.tight_layout(); plt.show()

## d+Au 200 GeV: the same comparison, PHOBOS Fig. 27

The bulk-tune screening's d+Au samples with this chain (`/data/OO-RHIC_bulk`, 10k minimum-bias events
each, b < 12 fm, bulk hadrons only): `dau_tau0424` (the adopted RHIC tune) and `dau_map` (the
2601.17234 MAP block). PHOBOS's points of Fig. 27 (`refs/phobos_dau200/`; 1σ band), η relative to
the deuteron beam, the Au-going side at η < 0. Model classes: the charged multiplicity on the Au
side, −5.2 < η < −3 (PHOBOS used its ring counters at 3 < |η| < 5.4), as fractions of the events;
the ⟨N_part⟩ per class is printed against PHOBOS's Table III.

In [ ]:
PD = json.loads(Path("../refs/phobos_dau200/phobos_dau200_dndeta.json").read_text())["bins"]
DAU_CL = [("0-20", 0, 20), ("20-40", 20, 40), ("40-60", 40, 60), ("60-80", 60, 80), ("80-100", 80, 100)]
DAU_NP = {"0-20": 15.5, "20-40": 10.8, "40-60": 7.1, "60-80": 4.3, "80-100": 2.7}
AXD = hist.axis.Regular(52, -5.2, 5.2, name="eta", label=r"$\eta$ (d-going > 0)")
def load_dau(name):
    rows, npart = [], []
    for bf in sorted(glob.glob(f"/data/OO-RHIC_bulk/{name}/*_bulk_bg.h5")):
        with h5py.File(bf) as h:
            n = int(h.attrs["nunits_written"]); pb, cb = B._units(h, n); npart.append(h["units/glauber_npart"][:n])
        ev = np.repeat(np.arange(n), cb)
        h2 = hist.Hist(hist.axis.Integer(0, n, name="ev"), AXD).fill(ev=ev, eta=eta_of(pb))
        rows.append(h2.values())
    rows = np.concatenate(rows).astype(float); npart = np.concatenate(npart)
    ring = rows[:, AXD.centers < -3.0].sum(1) + np.random.default_rng(1).uniform(0, 1, len(rows))
    return rows / AXD.widths, npart, np.argsort(-ring)
DAU = {nm: load_dau(nm) for nm in ("dau_tau0424", "dau_map")}
fig, axs = plt.subplots(2, 5, figsize=(20, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.4]})
xd = AXD.centers
for j, (lab, a, b) in enumerate(DAU_CL):
    pp = PD[lab]["points"]; pe = np.array([q["eta"] for q in pp]); pv = np.array([q["value"] for q in pp])
    pl = np.array([q["sys_lo"] for q in pp]); ph_ = np.array([q["sys_hi"] for q in pp])
    axs[0, j].fill_between(pe, pv - pl, pv + ph_, color="0.6", alpha=0.35, lw=0, step="mid")
    axs[0, j].plot(pe, pv, "s", ms=3, mfc="none", color="k", label="PHOBOS (1σ band)")
    axs[1, j].fill_between(pe, 1 - pl / pv, 1 + ph_ / pv, color="0.6", alpha=0.35, lw=0, step="mid")
    line = []
    for nm, mk, c in (("dau_tau0424", "o", "C3"), ("dau_map", "^", "C0")):
        rows, npart, order = DAU[nm]; N = len(rows); sel = order[int(N * a / 100):int(N * b / 100)]
        m = rows[sel].mean(0); e = rows[sel].std(0) / np.sqrt(len(sel))
        axs[0, j].errorbar(xd, m, e, fmt=mk, ms=3, color=c, label=("RHIC tune" if nm == "dau_tau0424" else "MAP block") + f", <N_part> {npart[sel].mean():.1f}")
        r = np.interp(pe, xd, m) / pv
        axs[1, j].plot(pe, r, mk, ms=3, color=c)
        line.append(f"{'tune' if nm == 'dau_tau0424' else 'MAP '} " + " ".join(f"{s_:+.0f}:{np.mean(r[np.abs(pe - s_) < 0.15]):.2f}" for s_ in (-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5)))
    print(f"d+Au {lab:6s} (PHOBOS <N_part> {DAU_NP[lab]}), model/PHOBOS at eta:\n   " + "\n   ".join(line))
    axs[0, j].set_title(f"d+Au 200 GeV, {lab} %", fontsize=10); axs[0, j].legend(fontsize=6.5); axs[0, j].set_ylim(0, None)
    axs[1, j].axhline(1, color="k", lw=0.8); axs[1, j].set_ylim(0.4, 1.8); axs[1, j].set_xlabel(r"$\eta$ (Au-going < 0)")
axs[0, 0].set_ylabel(r"$dN_{ch}/d\eta$"); axs[1, 0].set_ylabel("model / PHOBOS")
fig.suptitle("Figure 3. d+Au: the RHIC bulk tune and the MAP block against PHOBOS (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()

## Reading (2026-11-25; bulk hadrons only; model results, not vetted)

**Au+Au, the RHIC bulk tune** (150 events per class):
- The classes match PHOBOS's (⟨N_part⟩ 345 / 346, 74 / 73.5); N_ch(|η| < 5.4) is 1.00 and 0.99 × PHOBOS.
- Mid-rapidity 1.04–1.06 × PHOBOS in both classes, inside its 90 % band. The centrality dependence
  is about right: per participant pair 3.92 / 3.41 against PHOBOS's 3.79 / 3.17 (ratio 1.15 vs 1.20).
- Too narrow: FWHM 7.2 vs 7.6 (0–6 %) and 7.6 vs 8.2 (40–50 %); 0.90–0.95 at |η| = 3–4.5.
- **The far forward region is low and gets worse toward peripheral:** 0.87–0.95 at |η| ≥ 5 in 0–6 %,
  0.63–0.73 in 40–50 %.

**Au+Au, the MAP block of 2601.17234 in this chain:**
- Centrality dependence too strong: mid-rapidity 1.15 × PHOBOS in 0–6 %, 0.95 in 40–50 %. Without
  shadowing every repeated collision makes a string; the RHIC tune's shadowing 0.45 is what
  fixes the centrality dependence in this chain.
- Width and the far forward region: about as in the RHIC tune (0.70–0.78 at |η| ≥ 5 in 40–50 %).
- In this chain the MAP does not reproduce its own calibration data (it was fitted with UrQMD
  and freeze-out at e_sw 0.52 GeV/fm³; ours: iSS at T 0.15 GeV, no afterburner).

**d+Au (Figure 3), 10k events per set:** ⟨N_part⟩ per class matches PHOBOS's Table III.
- RHIC tune: the total and mid-rapidity right in central (1.00–1.04), 1.05–1.13 toward peripheral;
  **the Au side at η = −4 … −1 is 1.07–1.23 high, the Au beam rapidity (η = −5) 0.67–0.80 low;
  the d side at η = +2 … +5 0.85–1.0.**
- MAP: 0.6–0.9 × PHOBOS nearly everywhere (that is why the tune moved away from it), but with a
  flatter Au side and the d-going forward region at 1.0–1.1.

**What it says about the parameters:**
- **Shadowing (≈ 0.45) sets the centrality dependence** of mid-rapidity in A+A, and the RHIC tune's
  value works for Au+Au. Lowering it (MAP: 0) overshoots central Au+Au.
- **The rapidity loss of the first collisions is probably too large in the RHIC tune**
  (ylossParam4At4 / At6 2.15 against the MAP's 1.685, variance 0.45 against 0.68): the d side and the
  Au side of d+Au and the width of Au+Au all point to string ends too close to mid-rapidity; the
  MAP's smaller loss gives the better d-side forward region. A value in between (≈ 1.9, variance
  ≈ 0.55–0.6) is the obvious test, with the normalisation then restored by BG or α_rem.
- **The deficit at the beam rapidities (|η| ≳ 5, growing toward peripheral) is in both sets** in
  Au+Au and on the Au side of d+Au, so the rapidity-loss retune did not cause it. Candidates:
  the remnants (remnant_energy_loss_fraction 0.536), the deposition at large η_s (σ_η, MUSIC's
  η_s grid ends at ±6), the particlization of the dilute forward fluid (iSS at T 0.15, no
  corona), and PHOBOS's spectator-related tails in peripheral collisions.
- **For O+O:** the forward estimators sit at 3.5 < |η| < 4.6 (MBD) and 2.1 < |η| < 5.1 (EPD),
  where the Au+Au and d+Au comparisons show 10–20 % deviations, larger toward peripheral. The
  selection bias works through exactly this region (B4), so its size carries this systematic.

## Is this bulk good enough for the Au+Au energy-loss calibration? Yes, at mid-rapidity

The calibration (tuning page, "A3 and the proposed runs") asks whether the O+O chain's energy-loss
setting reproduces jet and π⁰ suppression in central Au+Au. For jets at |η| < 1 this bulk is good enough:
- **Density where the jets travel.** Central Au+Au mid-rapidity dN/dη is 1.04–1.06 × PHOBOS, inside its
  90 % C.L. band (±7.5 %), with the right N_part and total multiplicity. With T ∝ (dN/dη)^(1/3) that is
  ≈ +1.5–2 % in T and, with q̂ ∝ T³, ≈ +5 % in q̂: slightly *more* quenching. Small against what is tested:
  the O+O model quenches 2–5 × too little, and the q̂ type (Type 2, α_s 0.25 vs Type 3, 0.3) moves q̂ by
  ≈ 20–45 %.
- **The forward problems do not reach the jets.** The beam-rapidity deficit and the d+Au asymmetry sit at
  |η| ≳ 3–5. Centrality for the calibration comes from b (or classes matched by N_part), not from forward
  estimators.
- **The hydro is the one calibrated on Au+Au:** MUSIC runs the 2601.17234 MAP's transport coefficients.

**Not checked by dN/dη, and stated with the result:**
- **The early temperature profile.** dN/dη fixes the total entropy, not how fast the medium builds. Here
  the strings feed MUSIC gradually from τ = 0.424 fm/c (energy loss from 0.40 fm/c); JETSCAPE's AA tune
  (2204.01163) used TRENTo + free streaming and τ₀ = 0.6 fm/c. In O+O the jets interact mostly at
  t ≈ 0.4–3.5 fm/c (`quench_diag.ipynb`), so this phase matters. It is a property of the chain, the same
  in Au+Au and O+O, so calibrating in Au+Au with this chain absorbs it consistently.
- **Freeze-out** at T = 0.15 GeV instead of e_sw = 0.52 GeV/fm³: it changes when the fluid particlizes, but
  the energy loss stops at T_c = 0.16 GeV either way.
- **⟨p_T⟩ and flow (v₂) in Au+Au** are not checked; they matter for path-length dependence more than for the
  inclusive R_AA. A mid-rapidity ⟨p_T⟩ and dN/dη check in 0–10 % rides along with the first calibration run.

**Systematic to quote:** "Au+Au bulk at mid-rapidity within 4–6 % of PHOBOS (central), ≈ +5 % in q̂;
early-time medium build-up as in the O+O chain, not tuned."

## Proposed runs and run time

**Chain and settings for every Au+Au sample:** the O+O physics-run chain (`OO_MCGlauber_MUSIC_isr_rhictune.xml`)
with `--xml-set` Au + Au, generated nucleon positions, MUSIC grid ±13 fm (`IS/grid_max_x/y` 13; no grid-boundary
hit in the tests), `--b-max 4.7` (0–10 %), `--parton-ymax 1.0 --parton-y-mode any`, partons and hadronization input
stored, no hydro file, `--bulk-hadrons 1`. p+p reference: the existing `pp_ref` (70k events, the same chain).
Output root: `/data/AuAu-RHIC_phys` (new; nothing in the O+O production is touched). Each GPU step needs a go.

**Throughput (measured 2026-11-25, 2 × RTX 3090):** 15 s per event per job with 10 jobs per GPU (20 in all);
35 s at 16 per GPU (32 in all), where the host's CPU saturates. Campaign setting: 10–12 jobs per GPU, OMP 2–3,
`MUSIC_CUDA_SYNC=block`, MPS, the LBT table cache, staggered starts, 45-event jobs: **≈ 3,000–3,500 events/h**
sustained (an estimate between the two tests). Disk ≈ 0.4 MB per event.

| Run | Samples (`oo_bias.py`) | Energy-loss setting | p̂_T windows × events | Events | Time (both GPUs) | Status |
| --- | --- | --- | --- | --- | --- | --- |
| B1 | group `auau_b1`: `auau_med_w*` | current: Matter type 1 (2204.01163's Type 2), α_s 0.25, Lbt 0.25 | 10 windows 5–80 GeV × 2k (400 jobs of 50) | 20k | ≈ 7.5 h | **started 2026-11-25 21:16**; expected end ≈ 04:30–05:00 |
| B2 | group `auau_b2`: `auau_t3_w*` | 2204.01163's default Type 3: type 5, α_s 0.3, qhatA 10, qhatB 100, Lbt α_s 0.3 | the same, the same seeds | 20k | ≈ 7.5 h | starts automatically when B1 ends (`chain_b2.sh`, if B1 has ≤ 20 failed jobs; the user's go) |
| D9a / D9b | `oo_med` variants | the setting(s) the Au+Au data favour, on the O+O medium arm | 4 windows | 2 × 20k | ≈ 45 min each | after B1 / B2, on a go |

- **One setting after the other, not in parallel:** the first data verdict comes after ≈ 6–7 h instead of
  ≈ 12–14 h, and the next setting can follow from it. B2 is fixed to Type 3 for the night (the user's go);
  if B1 shows a large gap, Type 2 at α_s 0.35–0.40 follows after the review.
- **Windows in order (`--sequential`):** 23–30, 30–40, 40–55, 55–80 GeV first (≈ 1.3 h: the per-parton jet
  loss, comparable with O+O's 2–5 % in `quench_diag.ipynb`, against `oo_vac` as the vacuum reference, so no
  Au+Au vacuum arm); then 9–23 GeV (≈ 4 h: PHENIX π⁰ 8–20 GeV and most of STAR's charged jets 10–30 GeV);
  then 5–9 GeV (the low-p_T bins of the stitching).
- **Command** (the LBT table cache and staggered starts are new options of `oo_bias_docker.sh` / `oo_bias.py`):
  `JOBS_PER_GPU=10 OMP_THREADS=3 LBT_CACHE=~/prod_test/lbt_cache/lbt_tables_v1.bin ./oo_bias_docker.sh --dev /data/AuAu-RHIC_phys auau_b1 --sequential --stagger 12`
- **The data comparison** is prepared in `auau_jet_raa.ipynb` (STAR charged-jet R_AA, PHENIX π⁰ R_AA in central
  Au+Au; charged pions as the π⁰ proxy, since the jet files hold decayed π⁰), runnable on partial output.
- **Measured in B1:** ≈ 27 s per event per job with 10 jobs per GPU (all hadrons and the partons stored,
  3 OpenMP threads each: 60 threads on 48 cores, host CPU ≈ 95 %), so ≈ 2,600 events/h on both GPUs, against
  the 15 s of the charged-only throughput test.
- **Total:** B1 + B2 ≈ 15 h (B2 ends ≈ noon on 2026-11-26); D9a/b ≈ 1.5 h.